<a href="https://www.kaggle.com/code/josephawad91/agent-sustaniac?scriptVersionId=363877588" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🧭 Project Overview — Agent Sustaniac

Agent Sustaniac is an intelligent multi-agent system designed to help organizations quickly generate tailored sustainability roadmaps using AI. Many companies—especially small and medium-sized ones—struggle to understand where to begin with sustainability, what actions to prioritize, and how to translate high-level goals into practical, measurable steps. Traditional consulting is slow, expensive, and inaccessible for many firms. Sustaniac bridges that gap.

This project leverages the Google AI Agents framework, Gemini models, and structured tool calling to build an autonomous workflow that analyzes a company’s profile, identifies high-impact sustainability opportunities, and produces actionable short-, medium-, and long-term recommendations.

The system is composed of four specialized agents:  
- **Intake Agent** – Parses raw human descriptions into structured company profiles.  
- **Research Agent** – Analyzes the company’s industry, regulatory environment, and sustainability challenges.  
- **Strategy Agent** – Synthesizes the profile + research into a 3-year sustainability roadmap.  
- **Reporting Agent** – Generates a polished, human-readable sustainability report.  

The notebook demonstrates single-turn and multi-turn interactions, tool integration (best practices, KPIs, emissions estimation), and a long-running job example to simulate complex analysis workloads.  
A complete architectural diagram is included, along with reproducible code, commentary, and explanations of design decisions.

Agent Sustaniac shows how AI agents can automate complex expert workflows and lower the barrier for companies seeking guidance on sustainability planning—turning hours of manual work into seconds of intelligent, adaptive analysis.


📘 1. Introduction

Welcome to Sustaniac, an intelligent multi-agent sustainability assessment and strategy system built using the Google AI Developer Kit (ADK).
This notebook demonstrates:

How to create specialized AI agents

How to build tool-augmented workflows

How to orchestrate multi-step sustainability assessments

How to support multi-turn conversational interaction with memory

How to run long-running sustainability evaluations asynchronously

How to store, resume, and retrieve deep sustainability reports

This project simulates a real-world sustainability consultancy workflow, with agents collaborating to collect data, research impacts, generate strategies, and produce reports.

🧠 2. System Architecture Overview

Our system contains four core agents, each responsible for one domain:

1️⃣ Intake Agent — Company Profile Interpreter

Extracts structured sustainability-relevant metadata from raw user text.

2️⃣ Research Agent — Industry & Impact Analysis

Analyzes industry factors, environmental impacts, risks, regulations, etc.

3️⃣ Strategy Agent — Roadmap Generation

Creates detailed sustainability roadmaps including:

Quick wins

Medium-term actions

Long-term strategic moves

KPIs

Priorities & reasoning

4️⃣ Reporting Agent — Final Assessment Writer

Transforms all structured analysis into a polished, user-friendly report.

5️⃣ Orchestrator Agent — Pipeline Controller

Routes messages between agents and tools, manages flow, composes final results.

🔧 Tools Integrated

Best Practices Tool

KPI Suggestion Tool

Emissions Estimation Tool

🏗️ 3. Setup & Configuration

In this section, we load required packages, initialize system constants, and configure the model version (gemini-2.5-flash-lite), temperature, and retry logic.

We also configure InMemorySessionService and InMemoryMemoryService to support:

multi-turn conversations

stored agent state

resumable long-running workflows

In [ ]:
# 🔑 Configure Gemini API key from Kaggle Secrets

import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        "🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' "
        "to your Kaggle secrets. Details:", e
    )


In [ ]:
# ⚙️ Import ADK components and core dependencies

from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import AgentTool, FunctionTool, ToolContext
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search  # May be used for research
# from google.adk.code_executors import BuiltInCodeExecutor  # Uncomment if we need code execution later

print("✅ ADK components imported successfully.")


This section defines the core LLM agents using Google ADK LlmAgent:

Sustaniac Base Agent

Intake Agent

Research Agent

Strategy Agent

Reporting Agent

In [ ]:
# 🤖 Initialize the base Sustaniac agent using the ADK Agent class

def build_base_agent():
    """
    Creates a simple base agent using the ADK Agent class.
    This agent will be the core building block for more specialized
    agents (Intake, Research, Strategy, Reporting).
    """
    try:
        base_agent = Agent(
            name="sustaniac_base_agent",
            model="gemini-2.5-flash-lite",  # same style as course notebooks
            description="A base agent for sustainability consulting.",
            instruction=(
                "You are Sustaniac, an AI system dedicated to helping companies "
                "improve sustainability through research, best practices, and "
                "actionable roadmaps. For now, you are a simple test agent."
            ),
            tools=[],  # we'll add tools later
        )

        print("✅ Base Sustaniac agent created successfully.")
        return base_agent

    except Exception as e:
        print("❌ Error creating the base agent:", e)
        return None


base_agent = build_base_agent()



In [ ]:
# 🧪 Test the Sustaniac base agent using InMemoryRunner

if base_agent is None:
    print("❌ Base agent is not available. Fix previous errors before continuing.")
else:
    try:
        runner = InMemoryRunner(agent=base_agent)
        print("✅ Runner created.")

        test_prompt = "Hello Sustaniac! Briefly explain your mission in 2–3 sentences."
        response = await runner.run_debug(test_prompt)

        print("🟩 Agent Response:\n")
        print(response)

    except Exception as e:
        print("❌ Error running the base agent test:", e)


🧩 5. Implementing the Four Specialised Agents
5.1 Intake Agent

Goal: convert free-text business descriptions into structured sustainability profiles.

Produces:
> {
  "name": "...",
  "industry": "...",
  "company_size": "...",
  "locations": [...],
  "operations_summary": "...",
  "has_physical_facilities": true/false,
  "current_initiatives": [...],
  "time_horizon_years": number
}
> 

In [ ]:
# 🧱 Simplified CompanyProfile schema (clean and Kaggle-friendly)

from typing import TypedDict, List, Literal, Optional


class CompanyProfile(TypedDict, total=False):
    """
    Simplified structured representation of a company's information for sustainability analysis.
    """

    # Core identity
    name: str
    industry: str
    company_size: Literal["micro", "small", "medium", "large"]
    locations: List[str]

    # Operations overview
    operations_summary: str  # Short description of main activities
    has_physical_facilities: bool
    has_vehicle_fleet: bool

    # Sustainability current status
    current_initiatives: List[str]

    # Goals & constraints
    goals: List[str]
    budget_level: Optional[Literal["low", "medium", "high"]]
    time_horizon_years: Optional[int]


def show_company_profile_schema():
    """
    Helper for reminding what fields are expected.
    """
    print(
        """
Simplified CompanyProfile JSON format:

{
  "name": "ExampleCorp",
  "industry": "retail",
  "company_size": "medium",
  "locations": ["Spain", "France"],

  "operations_summary": "3 stores + small warehouse",
  "has_physical_facilities": true,
  "has_vehicle_fleet": false,

  "current_initiatives": ["LED lighting", "basic recycling"],

  "goals": ["reduce energy cost", "improve brand image"],
  "budget_level": "medium",
  "time_horizon_years": 3
}
"""
    )


print("✅ Simplified CompanyProfile schema defined.")


In [ ]:
# 🔧 Updated Strict Intake Agent — Enforces EXACT simplified schema keys

from google.adk.agents import Agent

def build_intake_agent():
    """
    Intake Agent that ALWAYS outputs JSON matching the simplified CompanyProfile schema.
    """
    try:
        intake_instruction = """
You are the Intake Agent for Sustaniac, an AI sustainability consultant.

Your task:
Convert a natural language company description into a JSON object that matches
the EXACT simplified CompanyProfile schema.

STRICT OUTPUT RULES:
- ALWAYS respond with ONLY valid JSON.
- NEVER include backticks, markdown, comments, explanations, or text outside JSON.
- Use ONLY these exact keys:

  "name": string
  "industry": string
  "company_size": "micro" | "small" | "medium" | "large"
  "locations": [string]

  "operations_summary": string
  "has_physical_facilities": boolean
  "has_vehicle_fleet": boolean

  "current_initiatives": [string]

  "goals": [string]
  "budget_level": "low" | "medium" | "high" | null
  "time_horizon_years": number | null

OTHER RULES:
- If something is missing, infer it reasonably.
- Use lowercase for industry names.
- Use lowercase for company_size and budget_level.
- goals MUST contain at least 1 inferred or explicit goal.
- current_initiatives MUST be an array, even if empty.
- locations MUST be an array with at least one entry (infer country from description if needed).
- operations_summary MUST be a short human-readable summary.
- boolean fields MUST be true or false, not strings.

Return ONLY a single valid JSON object.
""".strip()

        intake_agent = Agent(
            name="sustaniac_intake_agent",
            model="gemini-2.5-flash-lite",
            description="Strict intake agent that extracts company info and outputs structured JSON.",
            instruction=intake_instruction,
            tools=[],
        )

        print("✅ Strict Intake Agent created successfully.")
        return intake_agent

    except Exception as e:
        print("❌ Error creating the intake agent:", e)
        return None


# Replace old intake_agent
intake_agent = build_intake_agent()


In [ ]:
# 🧪 Test the strict Intake Agent

import json
import re
from google.adk.runners import InMemoryRunner

if intake_agent is None:
    print("❌ Intake agent not available.")
else:
    try:
        runner = InMemoryRunner(agent=intake_agent)
        print("✅ Intake runner ready.")

        sample_description = """
We are a mid-sized retail company in Spain with 3 clothing stores and 
one small warehouse. We don't own delivery trucks. We generate a lot of 
packaging waste. We already use LED lighting. Our goal is to reduce 
energy costs and improve sustainability over the next 3 years. 
Budget is moderate.
"""

        events = await runner.run_debug(sample_description)

        last_event = events[-1]
        raw_text = last_event.content.parts[0].text

        print("🟩 Raw output:\n", raw_text)

        cleaned = re.sub(r"```json|```", "", raw_text).strip()

        try:
            profile = json.loads(cleaned)
            print("\n✅ Parsed CompanyProfile:", profile)
            print("\n🔑 Keys:", list(profile.keys()))
        except Exception as json_error:
            print("❌ JSON Parsing Error:", json_error)
            print("Cleaned text was:\n", cleaned)

    except Exception as agent_error:
        print("❌ Error running Intake Agent:", agent_error)


In [ ]:
# 🧱 ResearchSummary schema - what the Research Agent should output

from typing import TypedDict, List


class ResearchSummary(TypedDict, total=False):
    """
    High-level sustainability research summary for a given company profile.
    """

    industry_summary: str
    key_impact_areas: List[str]          # e.g. ["energy", "waste", "logistics"]
    recommended_focus_areas: List[str]   # e.g. ["energy efficiency", "waste reduction"]
    example_best_practices: List[str]    # short list of practical examples
    risks_and_opportunities: str         # narrative summary
    notes_on_regulations: str            # high-level hints, not legal advice


def show_research_summary_schema():
    print(
        """
ResearchSummary JSON example:

{
  "industry_summary": "Retail companies often have impacts from stores, warehousing, and packaging.",
  "key_impact_areas": ["energy", "waste"],
  "recommended_focus_areas": ["energy efficiency", "packaging waste reduction"],
  "example_best_practices": [
    "Switch to LED lighting",
    "Optimize HVAC schedules",
    "Use recyclable or reusable packaging"
  ],
  "risks_and_opportunities": "Rising energy prices create risk; visible green actions can improve brand image.",
  "notes_on_regulations": "May be subject to EU ESG/CSRD reporting depending on size and listing status."
}
"""
    )


print("✅ ResearchSummary schema defined.")


5.2 Research Agent

Uses context + tool outputs to produce environmental analysis:

industry summary

impact areas

recommended focus areas

best practices

risk & opportunity analysis

regulation notes

In [ ]:
# 🔍 Build the Research Agent (Sustainability Analyst)

from google.adk.agents import Agent
from google.adk.tools import google_search

def build_research_agent():
    """
    Creates a Research Agent that takes a CompanyProfile JSON string as input
    and outputs a ResearchSummary JSON object.
    """
    try:
        tools = []
        try:
            # Attach google_search tool if available
            tools = [google_search]
            print("✅ Attached google_search tool to Research Agent.")
        except Exception as tool_error:
            print("⚠️ Could not attach google_search tool, continuing without it:", tool_error)

        research_instruction = """
You are the Research Agent for Sustaniac, an AI sustainability consultant.

INPUT:
- You receive a single JSON object that follows the CompanyProfile schema
  with these keys:
  - name, industry, company_size, locations
  - operations_summary, has_physical_facilities, has_vehicle_fleet
  - current_initiatives, goals, budget_level, time_horizon_years

YOUR TASK:
- Analyze the profile.
- (Optionally) call tools like google_search to gather general best practices
  for that industry and impact areas, if available.
- Produce a ResearchSummary JSON object with EXACT keys:

  "industry_summary": string
  "key_impact_areas": [string]
  "recommended_focus_areas": [string]
  "example_best_practices": [string]
  "risks_and_opportunities": string
  "notes_on_regulations": string

STRICT OUTPUT RULES:
- Respond with ONLY valid JSON, no markdown, no backticks, no explanations.
- If you cannot use tools, rely on your own knowledge.
- key_impact_areas and recommended_focus_areas MUST each have at least 1 item.
- example_best_practices MUST have at least 3 short, concrete actions.
- notes_on_regulations should be high-level and non-legal (e.g., "may be subject to EU reporting rules").
""".strip()

        research_agent = Agent(
            name="sustaniac_research_agent",
            model="gemini-2.5-flash-lite",
            description="Analyzes a company profile and produces a sustainability research summary.",
            instruction=research_instruction,
            tools=tools,
        )

        print("✅ Research Agent created successfully.")
        return research_agent

    except Exception as e:
        print("❌ Error creating the Research Agent:", e)
        return None


research_agent = build_research_agent()


In [ ]:
# 🧪 Test pipeline: Intake Agent -> Research Agent

import json
import re
from google.adk.runners import InMemoryRunner

if intake_agent is None or research_agent is None:
    print("❌ Intake or Research agent not available. Check previous cells.")
else:
    try:
        # 1️⃣ Run Intake Agent
        intake_runner = InMemoryRunner(agent=intake_agent)
        print("✅ Intake runner ready.")

        sample_description = """
We are a mid-sized retail company in Spain with 3 clothing stores and 
one small warehouse. We don't own delivery trucks. We generate a lot of 
packaging waste. We already use LED lighting. Our goal is to reduce 
energy costs and improve sustainability over the next 3 years. 
Budget is moderate.
"""

        intake_events = await intake_runner.run_debug(sample_description)
        intake_last_event = intake_events[-1]
        intake_raw = intake_last_event.content.parts[0].text
        intake_cleaned = re.sub(r"```json|```", "", intake_raw).strip()

        try:
            company_profile = json.loads(intake_cleaned)
            print("\n✅ CompanyProfile parsed:")
            print(company_profile)
        except Exception as e:
            print("❌ Error parsing CompanyProfile JSON:", e)
            raise e

        # 2️⃣ Run Research Agent with CompanyProfile JSON string as input
        research_runner = InMemoryRunner(agent=research_agent)
        print("\n✅ Research runner ready.")

        research_input = json.dumps(company_profile)

        research_events = await research_runner.run_debug(research_input)
        research_last_event = research_events[-1]
        research_raw = research_last_event.content.parts[0].text
        print("\n🟩 Raw Research Agent output:\n", research_raw)

        research_cleaned = re.sub(r"```json|```", "", research_raw).strip()

        try:
            research_summary = json.loads(research_cleaned)
            print("\n✅ Parsed ResearchSummary:")
            print(research_summary)
            print("\n🔑 ResearchSummary keys:", list(research_summary.keys()))
        except Exception as e:
            print("❌ Error parsing ResearchSummary JSON:", e)
            print("Cleaned text was:\n", research_cleaned)

    except Exception as pipeline_error:
        print("❌ Error in Intake -> Research pipeline:", pipeline_error)


In [ ]:
# 🧱 RoadmapSummary schema - the main output of the Strategy Agent

from typing import TypedDict, List, Literal, Optional


class RoadmapSummary(TypedDict, total=False):
    """
    Actionable sustainability roadmap generated using company profile + research summary.
    """

    overview: str  # High-level summary of recommended strategy

    quick_wins: List[str]  # Actions achievable in weeks–months
    medium_term_actions: List[str]  # 1–2 year improvements
    long_term_actions: List[str]  # multi-year strategic actions

    kpis: List[str]  # Key indicators the company should track

    impact_level: Literal["low", "medium", "high"]  # Expected overall sustainability impact
    priority_recommendations: List[str]  # Top 3–5 actions to prioritize

    reasoning_notes: str  # short narrative explaining why these recommendations were selected


def show_roadmap_summary_schema():
    print(
        """
RoadmapSummary JSON example:

{
  "overview": "Retail company should focus on energy efficiency and packaging waste reduction.",
  
  "quick_wins": [
    "Optimize HVAC schedules",
    "Switch remaining lighting to LED",
    "Improve recycling signage in stores"
  ],

  "medium_term_actions": [
    "Adopt reusable or recyclable packaging",
    "Implement waste audits",
    "Upgrade heating/cooling equipment"
  ],

  "long_term_actions": [
    "Install rooftop solar on warehouses",
    "Transition logistics partners to greener options"
  ],

  "kpis": [
    "kWh per square meter",
    "tons of waste per month",
    "percentage of recyclable packaging"
  ],

  "impact_level": "high",

  "priority_recommendations": [
    "Replace all lighting with LED",
    "Conduct an energy audit",
    "Switch to recyclable packaging"
  ],

  "reasoning_notes": "Actions chosen due to high energy cost and significant packaging waste."
}
"""
    )


print("✅ RoadmapSummary schema defined.")


5.3 Strategy Agent

Produces a structured roadmap:

quick wins

medium-term actions

long-term strategic recommendations

prioritized tasks

KPIs and impact level

In [ ]:
# 🧠 Build the Strategy / Roadmap Agent

from google.adk.agents import Agent

def build_strategy_agent():
    """
    Strategy Agent that turns a CompanyProfile + ResearchSummary JSON bundle
    into a structured RoadmapSummary JSON.
    """
    try:
        strategy_instruction = """
You are the Strategy Agent for Sustaniac.

INPUT:
You will receive a JSON object with TWO fields:

{
  "company_profile": {...},
  "research_summary": {...}
}

Your task:
Create a sustainability roadmap following the EXACT RoadmapSummary JSON schema:

{
  "overview": string,

  "quick_wins": [string],
  "medium_term_actions": [string],
  "long_term_actions": [string],

  "kpis": [string],

  "impact_level": "low" | "medium" | "high",
  "priority_recommendations": [string],

  "reasoning_notes": string
}

STRICT OUTPUT RULES:
- ALWAYS output ONLY valid JSON.
- NO markdown, NO comments, NO backticks.
- quick_wins MUST have at least 3 items.
- medium_term_actions MUST have at least 3 items.
- long_term_actions MUST have at least 2 items.
- priority_recommendations MUST contain top 3–5 actions.
- kpis MUST be actionable and measurable (e.g., "kWh/m2", "waste kg/month").
- impact_level must reflect expected overall improvement.

Tailor the plan to the company’s industry, size, goals, facilities, and operations.
""".strip()

        strategy_agent = Agent(
            name="sustaniac_strategy_agent",
            model="gemini-2.5-flash-lite",
            description="Creates an actionable sustainability roadmap.",
            instruction=strategy_instruction,
            tools=[],  # no tools yet (we'll add KPI & emissions tools later)
        )

        print("✅ Strategy Agent created successfully.")
        return strategy_agent

    except Exception as e:
        print("❌ Error creating Strategy Agent:", e)
        return None


strategy_agent = build_strategy_agent()


In [ ]:
# 🧪 Test pipeline: Research -> Strategy Agent

import json
import re
from google.adk.runners import InMemoryRunner

if research_agent is None or strategy_agent is None:
    print("❌ Missing research or strategy agent.")
else:
    try:
        # 1️⃣ Run Intake -> Research pipeline first
        intake_runner = InMemoryRunner(agent=intake_agent)
        research_runner = InMemoryRunner(agent=research_agent)
        strategy_runner = InMemoryRunner(agent=strategy_agent)

        print("🏁 Runners ready.")

        # Inputs
        company_description = """
We are a mid-sized retail company in Spain with 3 clothing stores and one small warehouse.
No vehicle fleet. Packaging waste is high. LED lighting installed in two stores.
Goal is to reduce energy costs and improve sustainability over the next 3 years.
Budget is moderate.
"""

        # Step 1: Intake
        intake_events = await intake_runner.run_debug(company_description)
        intake_raw = intake_events[-1].content.parts[0].text
        intake_clean = re.sub(r"```json|```", "", intake_raw).strip()
        company_profile = json.loads(intake_clean)

        print("✅ CompanyProfile loaded.")

        # Step 2: Research
        research_input = json.dumps(company_profile)
        research_events = await research_runner.run_debug(research_input)
        research_raw = research_events[-1].content.parts[0].text
        research_clean = re.sub(r"```json|```", "", research_raw).strip()
        research_summary = json.loads(research_clean)

        print("✅ ResearchSummary loaded.")

        # Step 3: Strategy
        strategy_input_json = json.dumps({
            "company_profile": company_profile,
            "research_summary": research_summary
        })

        strategy_events = await strategy_runner.run_debug(strategy_input_json)
        strategy_raw = strategy_events[-1].content.parts[0].text
        print("\n🟩 Raw Strategy output:\n", strategy_raw)

        strategy_clean = re.sub(r"```json|```", "", strategy_raw).strip()
        roadmap = json.loads(strategy_clean)

        print("\n✅ Parsed RoadmapSummary:")
        print(roadmap)
        print("\n🔑 Keys:", list(roadmap.keys()))

    except Exception as e:
        print("❌ Error in Research -> Strategy pipeline:", e)


5.4 Reporting Agent

Creates a polished, narrative final report suitable for business stakeholders.

In [ ]:
# 📝 Build the Reporting Agent - Generates a human-readable sustainability report

from google.adk.agents import Agent

def build_reporting_agent():
    """
    Creates an agent that turns a RoadmapSummary JSON object into a clean,
    narrative sustainability report for human readers.
    """
    try:
        reporting_instruction = """
You are the Reporting Agent for Sustaniac, an AI sustainability consultant.

INPUT:
You will receive a JSON object that follows the RoadmapSummary schema:

{
  "overview": string,
  "quick_wins": [string],
  "medium_term_actions": [string],
  "long_term_actions": [string],
  "kpis": [string],
  "impact_level": "low" | "medium" | "high",
  "priority_recommendations": [string],
  "reasoning_notes": string
}

TASK:
Transform this content into a clean, readable sustainability report.
Use a professional tone with clear structure.

STRUCTURE:
1. **Sustainability Overview**
2. **Top Priorities**
3. **Quick Wins (0–6 months)**
4. **Medium-Term Actions (6–24 months)**
5. **Long-Term Strategic Actions (2+ years)**
6. **Key Performance Indicators**
7. **Expected Impact Level**
8. **Notes & Reasoning**

RULES:
- Only output natural language text.
- No JSON.
- No markdown code fences.
- No commentary about the agent itself.
""".strip()

        reporting_agent = Agent(
            name="sustaniac_reporting_agent",
            model="gemini-2.5-flash-lite",
            description="Generates a human-readable sustainability plan from structured roadmap data.",
            instruction=reporting_instruction,
            tools=[],
        )

        print("✅ Reporting Agent created successfully.")
        return reporting_agent

    except Exception as e:
        print("❌ Error creating Reporting Agent:", e)
        return None


reporting_agent = build_reporting_agent()


In [ ]:
# 🧪 Test pipeline: Strategy -> Reporting

import json
import re
from google.adk.runners import InMemoryRunner

if strategy_agent is None or reporting_agent is None:
    print("❌ Missing strategy or reporting agent.")
else:
    try:
        # Already have company_profile and research_summary from Step 5 test.
        # If notebook restarted, re-run Step 5 first.

        strategy_runner = InMemoryRunner(agent=strategy_agent)
        reporting_runner = InMemoryRunner(agent=reporting_agent)

        print("🏁 Runners ready.")

        # Prepare combined input for Strategy Agent
        strategy_input = json.dumps({
            "company_profile": company_profile,
            "research_summary": research_summary
        })

        # Step 1: Strategy Agent
        strategy_events = await strategy_runner.run_debug(strategy_input)
        strategy_raw = strategy_events[-1].content.parts[0].text
        strategy_clean = re.sub(r"```json|```", "", strategy_raw).strip()
        roadmap = json.loads(strategy_clean)

        print("✅ Strategy roadmap loaded.")

        # Step 2: Reporting Agent
        reporting_input = json.dumps(roadmap)

        reporting_events = await reporting_runner.run_debug(reporting_input)
        final_report = reporting_events[-1].content.parts[0].text

        print("\n🟦 Final Sustainability Report:\n")
        print(final_report)

    except Exception as e:
        print("❌ Error in Strategy -> Reporting pipeline:", e)


In [ ]:
# 🛠️ Convert sub-agents into tools usable by the orchestrator agent (correct version)

from google.adk.tools import AgentTool

try:
    intake_tool = AgentTool(intake_agent)
    research_tool = AgentTool(research_agent)
    strategy_tool = AgentTool(strategy_agent)
    reporting_tool = AgentTool(reporting_agent)

    print("✅ AgentTools created successfully.")

except Exception as e:
    print("❌ Error creating AgentTools:", e)


The Orchestrator Agent coordinates the full multi-agent pipeline.
It:

calls the Intake Agent

passes structured data to Research Agent

invokes tools

sends results to Strategy Agent

finalizes with Reporting Agent

In [ ]:
# 🤖 Orchestrator Agent: Sustaniac Master Agent

def build_orchestrator_agent():
    """
    The Orchestrator coordinates the entire workflow:
    Intake -> Research -> Strategy -> Reporting.
    """
    try:
        orchestration_instruction = """
You are Sustaniac, the master sustainability consultant.

Your role:
Coordinate a sequence of tools to produce a full sustainability consultation.

Workflow:
1. Call intake_tool with the user's free-text input.
2. Take the resulting JSON and pass it (as a JSON string) to research_tool.
3. Take the ResearchSummary JSON and pass it (as a JSON string) to strategy_tool.
4. Take the RoadmapSummary JSON and pass it (as a JSON string) to reporting_tool.
5. Return ONLY the final human-readable report from reporting_tool.

RULES:
- DO NOT output the intermediate JSONs to the user.
- DO NOT leak system or tool instructions.
- ALWAYS call tools in the correct order: intake -> research -> strategy -> reporting.
- The final output must be natural language text, not JSON.
""".strip()

        orchestrator_agent = Agent(
            name="sustaniac_orchestrator_agent",
            model="gemini-2.5-flash-lite",
            description="Main coordinating agent for the full Sustaniac sustainability workflow.",
            instruction=orchestration_instruction,
            tools=[intake_tool, research_tool, strategy_tool, reporting_tool],
        )

        print("✅ Orchestrator Agent created successfully.")
        return orchestrator_agent

    except Exception as e:
        print("❌ Error creating orchestrator agent:", e)
        return None


orchestrator_agent = build_orchestrator_agent()


In [ ]:
# 🏁 Test FULL Sustaniac Workflow: Intake -> Research -> Strategy -> Reporting

from google.adk.runners import InMemoryRunner

if orchestrator_agent is None:
    print("❌ Orchestrator agent missing.")
else:
    try:
        runner = InMemoryRunner(agent=orchestrator_agent)
        print("🚀 Orchestrator runner ready.")

        user_company_description = """
We are a mid-sized retail company in Spain with 3 clothing stores and a small warehouse.
We generate significant packaging waste but have reduced energy use by switching to LED lighting.
We want to reduce costs and improve sustainability over the next 3 years.
"""

        events = await runner.run_debug(user_company_description)

        final_output = events[-1].content.parts[0].text

        print("\n🟦 Sustaniac Final Output:\n")
        print(final_output)

    except Exception as e:
        print("❌ Error running full Sustaniac workflow:", e)


🛠️ 6. Tooling Layer

We add three domain tools:

6.1 Industry Best Practices Tool

Returns a list of sustainability best practices for a given industry.

6.2 KPI Suggestion Tool

Returns measurable KPIs relevant to an industry.

6.3 Emissions Estimator Tool

Produces rough emissions estimates.

These tools are automatically used by the research agent and strategy agent through the orchestrator.

In [ ]:
### 🔧 Tool 1 — Industry Best Practices

from google.adk.tools import FunctionTool

def get_industry_best_practices(industry: str) -> dict:
    """
    Returns sustainability best practices for the given industry.
    Uses a simple lookup table to ensure Kaggle reliability.
    """
    try:
        if not isinstance(industry, str):
            return {"error": "industry must be a string"}

        industry = industry.lower().strip()

        database = {
            "retail": [
                "Switch all lighting to LED",
                "Optimize HVAC schedules",
                "Use recyclable or reusable packaging",
                "Improve waste segregation training",
                "Conduct regular waste audits"
            ],
            "manufacturing": [
                "Upgrade to energy-efficient machinery",
                "Recover and reuse heat in production lines",
                "Reduce water consumption",
                "Implement circular materials programs",
                "Install solar panels"
            ],
            "logistics": [
                "Optimize delivery routes",
                "Transition to hybrid or EV fleet",
                "Reduce packaging volume",
                "Ensure proper vehicle maintenance",
                "Optimize warehouse HVAC"
            ],
            "saas": [
                "Use green cloud providers",
                "Optimize compute workloads",
                "Recycle e-waste",
                "Track cloud carbon usage",
                "Train employees on sustainability"
            ]
        }

        practices = database.get(industry, [
            "Improve energy efficiency",
            "Reduce operational waste",
            "Enhance recycling workflows",
            "Train employees on sustainability"
        ])

        return {"industry": industry, "best_practices": practices}

    except Exception as e:
        return {"error": f"Exception: {str(e)}"}


best_practices_tool = FunctionTool(func=get_industry_best_practices)

print("✅ Created: industry_best_practices_tool")


In [ ]:
# 🛠️ Tool 2: KPI Suggestion Tool

def suggest_kpis(industry: str) -> dict:
    """
    Given an industry name, returns a list of recommended sustainability KPIs.
    """
    try:
        if not isinstance(industry, str):
            return {"error": "industry must be a string"}

        industry = industry.lower().strip()

        kpi_database = {
            "retail": [
                "kWh per square meter",
                "tons of waste per month",
                "percentage of recyclable packaging",
                "water usage per store"
            ],
            "manufacturing": [
                "energy intensity (kWh per unit)",
                "CO2 emissions per unit produced",
                "waste recycling rate",
                "water usage per production cycle"
            ],
            "logistics": [
                "fuel consumption per km",
                "CO2 per delivery",
                "delivery efficiency score",
                "vehicle utilization rate"
            ],
            "saas": [
                "cloud energy usage (kWh)",
                "CO2 per compute-hour",
                "server utilization",
                "hardware recycling volume"
            ]
        }

        return {
            "industry": industry,
            "kpis": kpi_database.get(industry, [
                "total energy usage (kWh)",
                "waste generated per month",
                "percentage of waste recycled"
            ])
        }

    except Exception as e:
        return {"error": f"Exception: {str(e)}"}


kpi_tool = FunctionTool(func=suggest_kpis)

print("✅ Created: kpi_suggestion_tool")


In [ ]:
# 🛠️ Tool 3: Emissions Estimator Tool

def estimate_emissions(operations_summary: str, has_vehicle_fleet: bool) -> dict:
    """
    VERY rough emissions estimate. Not scientific—just for AI agent context.
    """
    try:
        if not isinstance(operations_summary, str):
            return {"error": "operations_summary must be a string"}

        if not isinstance(has_vehicle_fleet, bool):
            return {"error": "has_vehicle_fleet must be boolean"}

        base = 50  # baseline score for any company

        ops = operations_summary.lower()

        if "warehouse" in ops:
            base += 20
        if "store" in ops:
            base += 10

        if has_vehicle_fleet:
            base += 40

        return {
            "estimated_monthly_emissions_kg_co2e": base,
            "note": "Rough estimate for demonstration only."
        }

    except Exception as e:
        return {"error": f"Exception: {str(e)}"}


emissions_tool = FunctionTool(func=estimate_emissions)

print("✅ Created: emissions_estimator_tool")


In [ ]:
# Step 8.2 — Clear tools from Research & Strategy agents 
# (because this model does NOT support automatic function calling)

try:
    research_agent.tools = []
    strategy_agent.tools = []
    print("✅ Cleared tools from Research & Strategy agents. We'll use tools manually.")
except Exception as e:
    print("❌ Error while clearing tools:", e)


In [ ]:
print("➡ Test best practices:", get_industry_best_practices("retail"))
print("➡ Test kpis:", suggest_kpis("retail"))
print("➡ Test emissions:", estimate_emissions("3 stores and a warehouse", False))


🔄 7. Orchestrator Agent

This agent coordinates the entire workflow:

Parse user message → send to intake agent

Send intake outputs + tools → research agent

Send research outputs + tools → strategy agent

Send everything → reporting agent

Return the final report

It acts like the “project manager” for the AI team.

In [ ]:
# Step 8.4 — Use tools + Strategy Agent together (manual integration demo)

from google.adk.runners import InMemoryRunner
import json

# 1️⃣ Example CompanyProfile
profile = {
    "name": "Retail Demo",
    "industry": "retail",
    "company_size": "medium",
    "locations": ["Spain"],
    "operations_summary": "3 stores and a small warehouse",
    "has_physical_facilities": True,
    "has_vehicle_fleet": False,
    "current_initiatives": ["LED lighting"],
    "goals": ["reduce energy cost"],
    "budget_level": "medium",
    "time_horizon_years": 3
}

# 2️⃣ Call tool functions
best_practices = get_industry_best_practices(profile["industry"])
kpi_suggestions = suggest_kpis(profile["industry"])
emissions_estimate = estimate_emissions(
    operations_summary=profile["operations_summary"],
    has_vehicle_fleet=profile["has_vehicle_fleet"],
)

print("🛠️ Tool outputs:")
print(best_practices)
print(kpi_suggestions)
print(emissions_estimate)

# 3️⃣ Create ResearchSummary (minimal demo version)
research_summary = {
    "industry_summary": "Retail companies often have energy + packaging waste issues.",
    "key_impact_areas": ["energy", "waste"],
    "recommended_focus_areas": ["energy efficiency", "packaging reduction"],
    "example_best_practices": best_practices["best_practices"],
    "risks_and_opportunities": "Rising energy costs.",
    "notes_on_regulations": "CSRD may apply depending on size."
}

# 4️⃣ Call Strategy Agent using tool outputs
strategy_input = {
    "company_profile": profile,
    "research_summary": research_summary,
    "tool_outputs": {
        "best_practices": best_practices,
        "kpi_suggestions": kpi_suggestions,
        "emissions_estimate": emissions_estimate
    }
}

strategy_runner = InMemoryRunner(agent=strategy_agent)
events = await strategy_runner.run_debug(json.dumps(strategy_input))

print("\n🟦 Strategy Agent Output using tool context:\n")
print(events[-1].content.parts[0].text)


We enable multi-turn conversations using ADK’s session + memory services.
This allows Sustaniac to remember previous company analyses and answer follow-up questions.

🔁 9. Multi-turn Interaction with Memory

Here we showcase how the system supports:

ongoing conversation

memory of previous agent outputs

contextually grounded follow-up answers

Example:

User:
“We are a mid-sized retail company with 3 stores…”

Our pipeline produces a full roadmap.

User:
“Thanks. What are the three lowest-cost actions we can start next month?”

The system recalls:

industry

footprint

previous recommendations

And responds with low-cost quick wins such as waste audits, reusable bag program, LED optimization.

This section proves our system supports conversational refinement.

In [ ]:
# Step 9.1
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str,
    session_id: str = "default"
):
    """Debug session that prints text from any event type and dumps raw events when needed."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve the session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # Loop messages
    for query in user_queries:
        print(f"\nUser > {query}")
        msg = types.Content(role="user", parts=[types.Part(text=query)])

        async for event in runner_instance.run_async(
            user_id=USER_ID,
            session_id=session.id,
            new_message=msg
        ):

            printed = False

            # 1) Standard text output
            if event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model > {text}")
                    printed = True

            # 2) If model response event exists
            if hasattr(event, "model_response_event") and event.model_response_event:
                mre = event.model_response_event
                if mre.content and mre.content.parts:
                    text = mre.content.parts[0].text
                    if text:
                        print(f"Model > {text}")
                        printed = True

            # 3) If nothing printed so far, dump raw event for debugging
            if not printed:
                print(f"[RAW EVENT] {event}")


In [ ]:
# Step 9.2 — Session + Memory using ADK Runner (course-correct pattern)

from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.runners import Runner

# Constants
APP_NAME = "sustaniac_app"
USER_ID = "demo_user_1"

# Services
session_service = InMemorySessionService()
memory_service = InMemoryMemoryService()

# Runner with full services
runner = Runner(
    app_name=APP_NAME,
    agent=orchestrator_agent,
    session_service=session_service,
    memory_service=memory_service,
)

print("✅ Runner with sessions & memory ready.")


In [ ]:
# Step 9.3 — Multi-turn test (session + memory)

await run_session(
    runner_instance=runner,
    user_queries=[
        """
We are a mid-sized retail company in Spain with 3 clothing stores and a small warehouse.
We generate significant packaging waste but have reduced energy consumption using LED lighting.
We want to reduce costs and improve sustainability over the next 3 years.
""",
        """
Thanks. What are the three lowest-cost actions we can start next month?
"""
    ],
    session_id="sustaniac_test"
)


🕒 10. Long-Running Sustainability Assessment (Asynchronous)

In this section, we demonstrate how to use:

start_long_running_job()

store job_id

simulate the user coming back later

resume_long_running_job(job_id)

retrieve a large, detailed sustainability report

This reproduces real-world consulting workflows where:

deep environmental analysis takes time

users disconnect and return later

In [ ]:
# Step 10.1 (patched) — Robust long-running assessment using Runner + sessions

import uuid
from typing import Dict, Any
from google.genai import types

# Keep LONG_RUNNING_JOBS and get_long_running_assessment as before

async def start_long_running_assessment(
    runner_instance: Runner,
    session_id: str,
    user_message: str,
) -> str:
    """
    Simulate starting a long-running sustainability assessment.
    Uses Runner + session_service and robustly extracts the final text
    from the event stream (content, model_response_event, or function_response.result).
    """

    print(f"\n=== START LONG-RUNNING ASSESSMENT (session: {session_id}) ===")

    # Create or retrieve session (same pattern as Step 9 helper)
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except Exception:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    msg = types.Content(role="user", parts=[types.Part(text=user_message)])

    final_text_candidates: list[str] = []

    # Stream events from the orchestrator agent
    async for event in runner_instance.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=msg,
    ):
        # 1️⃣ Direct text in event.content.parts
        if event.content and event.content.parts:
            part = event.content.parts[0]

            # Plain text
            if getattr(part, "text", None):
                txt = part.text
                if txt and txt != "None":
                    final_text_candidates.append(txt)

            # Function response with 'result' field
            fr = getattr(part, "function_response", None)
            if fr is not None:
                try:
                    resp = fr.response
                    if isinstance(resp, dict):
                        res = resp.get("result")
                        if isinstance(res, str) and res.strip():
                            final_text_candidates.append(res)
                except Exception:
                    pass

        # 2️⃣ Sometimes text lives inside model_response_event
        if hasattr(event, "model_response_event") and event.model_response_event:
            mre = event.model_response_event
            if mre.content and mre.content.parts:
                part = mre.content.parts[0]
                if getattr(part, "text", None):
                    txt = part.text
                    if txt and txt != "None":
                        final_text_candidates.append(txt)

    if not final_text_candidates:
        raise RuntimeError("No final text received from long-running assessment.")

    # Take the last candidate as the final detailed report
    final_text = final_text_candidates[-1]

    # Create a job_id and store the result
    job_id = str(uuid.uuid4())[:8]
    LONG_RUNNING_JOBS[job_id] = {
        "status": "completed",  # in a real system this would start as 'running'
        "session_id": session.id,
        "user_id": USER_ID,
        "detailed_report": final_text,
    }

    print(f"\n✅ Long-running assessment finished and stored under job_id: {job_id}")
    return job_id


Example: A medium-sized retail company in Spain.
Run the full pipeline via the orchestrator to generate an actionable sustainability roadmap.

In [ ]:
# Step 10.2 — Reliable start + resume of a long-running assessment (with retries)

import asyncio

# A reasonably long but not crazy prompt (to reduce overload risk)
deep_dive_query = """
We want a deep-dive sustainability assessment for our company.
Please analyze in detail:
- energy use
- waste generation
- supply chain emissions

Then produce a structured roadmap with:
- quick wins (0–6 months)
- medium-term actions (6–24 months)
- long-term actions (2+ years)
- KPIs
- expected impact level.

Assume this is a heavy analysis that may take a long time to compute.
"""

async def run_long_running_demo_with_retry(
    max_attempts: int = 5,
    delay_seconds: float = 2.0,
):
    global job_id  # so we can reuse it later if needed

    for attempt in range(1, max_attempts + 1):
        try:
            print(f"\n=== Attempt {attempt} to start long-running assessment ===")

            job_id = await start_long_running_assessment(
                runner_instance=runner,
                session_id="longrun_session_1",
                user_message=deep_dive_query,
            )

            print(f"\n✅ Long-running assessment started & stored with job_id: {job_id}")
            break

        except Exception as e:
            msg = str(e)
            # Check for typical transient errors
            if "503" in msg or "UNAVAILABLE" in msg or "overloaded" in msg or "429" in msg:
                if attempt < max_attempts:
                    print(f"⚠️ Model overloaded or unavailable (attempt {attempt}). Retrying in {delay_seconds}s...")
                    await asyncio.sleep(delay_seconds)
                    continue
                else:
                    print("❌ Reached maximum retry attempts. Failing.")
                    raise
            else:
                # Non-retryable error
                print("❌ Non-retryable error during long-running assessment:")
                raise

    # If we got here, job_id should exist
    print(f"\n🔁 Simulated 'pause' — user goes away and later returns with job_id: {job_id}")

    # "Resume" by fetching the stored result
    job_record = get_long_running_assessment(job_id)

    print("\n=== RESUME LONG-RUNNING ASSESSMENT ===")
    print("Job status:", job_record["status"])
    print("\n📝 Retrieved detailed report (truncated):\n")
    print(job_record["detailed_report"][:800], "...\n")


# Run the demo
try:
    await run_long_running_demo_with_retry()
except Exception as e:
    print("❌ Error in long-running assessment demo with retry:", e)


🏁 11. Final Notes & Architecture Summary

We successfully implemented:

Multi-agent sustainability consultancy

Tool-augmented reasoning

Automatic orchestration

Multi-turn memory support

Long-running resumable workflows

This system is modular and extendable to:

ESG scoring

CSRD-compliant reporting

Supply chain analysis

Carbon footprint modeling

Industry-specific sustainability frameworks

Future improvements may include:

persistent long-term memory

vector retrieval for regulations

agent collaboration visualizations

richer emissions modeling

🎉 12. Conclusion

This notebook demonstrates a full professional-grade multi-agent sustainability assessment system built entirely with Google’s ADK, including:

structured agents

tool integration

orchestration logic

memory + session management

long-running tasks with resumable execution

It reflects a real-world consulting workflow and showcases how advanced agents can work together to deliver expert sustainability insights.

**Future work**:

Add deployment (Cloud Run / Agent Engine)

Integrate real regulatory databases

Add industry-specific scoring models

Provide a business dashboard for reports